# IRRA + SCFP V2 — ICFG-PEDES / single T4

V2 only: semantic slots + adaptive **intermediate-feature** FFT. No V3 teacher/distillation.
Run top to bottom. Attach ICFG-PEDES and the verified IRRA checkpoint dataset.
Until `feat/scfp-v2` is pushed, attach the source ZIP made by `python -m experiments.scfp.bundle`.
Warm-start is an engineering diagnostic, with no validation/test selection. Scientific mode splits training identities **before** training from CLIP. Official test is opt-in.

In [ ]:
EXPERIMENT = "v2"  # baseline | slots | frequency | v2
TRAINING_PROTOCOL = "warmstart"  # warmstart | scientific
EPOCHS = 1
BATCH_SIZE = 64
SEED = 42
INIT_CHECKPOINT = None  # auto-discover a UNIQUE best.pth in Kaggle Input
ORIGINAL_CONFIG = None  # defaults to configs.yaml beside INIT_CHECKPOINT
RESUME_CHECKPOINT = None  # last.pth; keep original EPOCHS/schedule
BACKBONE_LR = 1e-6  # scientific screening: set 1e-5 for ALL variants
SCFP_LR = 1e-4
RUN_UNIT_TESTS = True
RUN_SMOKE_TEST = True
RUN_BASELINE_EVALUATION = False
RUN_FINAL_TEST = False
SOURCE_ZIP = None  # auto-discover an intact IRRA_scfp_v2_source*.zip
SOURCE_DIR = None  # auto-discover extracted source if Kaggle unpacked the ZIP
ANNOTATION_SOURCE = None
IMAGES_SOURCE = None
INPUT_ROOT = "/kaggle/input"
DATA_ROOT = "/kaggle/working/data"
RUN_ROOT = "/kaggle/working/SCFP_V2_runs"
REPO_URL = "https://github.com/thanhphu25/IRRA.git"
BRANCH = "feat/scfp-v2"
assert EXPERIMENT in {"baseline", "slots", "frequency", "v2"}
assert TRAINING_PROTOCOL in {"warmstart", "scientific"}
assert 1 <= EPOCHS <= 60


In [ ]:
import os, sys, subprocess, json, zipfile, hashlib, shutil
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
def run(command, cwd=None):
    print(" ".join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=cwd, check=True)
root = Path(INPUT_ROOT)
if SOURCE_ZIP is None:
    sources = sorted(root.rglob("IRRA_scfp_v2_source*.zip"))
    if len(sources) > 1:
        raise ValueError(f"Set SOURCE_ZIP explicitly: {sources}")
    SOURCE_ZIP = str(sources[0]) if sources else None
if SOURCE_ZIP is None and SOURCE_DIR is None:
    extracted = sorted({p.parent for p in root.rglob("source_manifest.json")
                        if (p.parent / "model/scfp/feature_frequency.py").is_file()
                        and (p.parent / "train.py").is_file()})
    if len(extracted) > 1:
        raise ValueError(f"Set SOURCE_DIR explicitly: {extracted}")
    SOURCE_DIR = str(extracted[0]) if extracted else None
REPO = Path("/kaggle/working/IRRA_scfp_v2")
if REPO.exists():
    raise FileExistsError(f"{REPO} already exists; use a fresh session or choose a fresh source directory")
if SOURCE_ZIP:
    REPO.mkdir()
    with zipfile.ZipFile(SOURCE_ZIP) as archive:
        for entry in archive.infolist():
            resolved = (REPO / entry.filename).resolve()
            if not resolved.is_relative_to(REPO.resolve()):
                raise ValueError("Unsafe ZIP path")
        archive.extractall(REPO)
elif SOURCE_DIR:
    shutil.copytree(SOURCE_DIR, REPO)
else:
    # The research branch must already exist remotely; this notebook never pushes.
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO])
manifest_file = REPO / "source_manifest.json"
if manifest_file.exists():
    for name, expected in json.loads(manifest_file.read_text()).items():
        source_file = (REPO / name).resolve()
        if not source_file.is_relative_to(REPO.resolve()):
            raise ValueError("Unsafe source manifest path")
        assert hashlib.sha256(source_file.read_bytes()).hexdigest() == expected, name
os.chdir(REPO)
sys.path.insert(0, str(REPO))
run([sys.executable, "-m", "pip", "install", "-q", "-r", "experiments/scfp/requirements.txt"])
import torch, torchvision
print({"python": sys.version, "torch": torch.__version__, "torchvision": torchvision.__version__,
       "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})
assert torch.cuda.is_available(), "Select a Kaggle GPU accelerator"
assert torch.cuda.device_count() == 1, "Use a single visible GPU"


In [ ]:
from experiments.scfp.prepare_kaggle import prepare_icfg, find_checkpoint
from utils.iotools import load_train_configs
prepared = prepare_icfg(INPUT_ROOT, DATA_ROOT, ANNOTATION_SOURCE, IMAGES_SOURCE)
if TRAINING_PROTOCOL == "warmstart" or RUN_BASELINE_EVALUATION:
    BASELINE_CHECKPOINT = str(find_checkpoint(INPUT_ROOT, INIT_CHECKPOINT))
    if TRAINING_PROTOCOL == "warmstart":
        INIT_CHECKPOINT = BASELINE_CHECKPOINT
    ORIGINAL_CONFIG = ORIGINAL_CONFIG or str(Path(BASELINE_CHECKPOINT).with_name("configs.yaml"))
    assert Path(ORIGINAL_CONFIG).is_file(), "Set ORIGINAL_CONFIG explicitly"
    original = load_train_configs(ORIGINAL_CONFIG)
    print("Original checkpoint:", BASELINE_CHECKPOINT)
    print("Original config:", ORIGINAL_CONFIG)
else:
    assert INIT_CHECKPOINT is None, "Scientific mode must start from CLIP"
if TRAINING_PROTOCOL == "scientific" and RESUME_CHECKPOINT is None:
    assert INIT_CHECKPOINT is None, "Do not initialize scientific comparison from trained IRRA"
RUNS = Path(RUN_ROOT)
RUNS.mkdir(parents=True, exist_ok=True)
from datetime import datetime
from zoneinfo import ZoneInfo
import uuid
session_name = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6]
SESSION = RUNS / ("checks_" + session_name)
SESSION.mkdir(exist_ok=False)


In [ ]:
if RUN_UNIT_TESTS:
    run([sys.executable, "-m", "pytest", "-q"])

labels = {"baseline": "A0_baseline", "slots": "A1_slots", "frequency": "A2_frequency", "v2": "V2_full"}
common = ["--scfp_mode", EXPERIMENT, "--dataset_name", "ICFG-PEDES", "--root_dir", DATA_ROOT,
          "--training_protocol", TRAINING_PROTOCOL, "--val_dataset", "none" if TRAINING_PROTOCOL == "warmstart" else "holdout",
          "--seed", str(SEED), "--batch_size", str(BATCH_SIZE), "--test_batch_size", "64", "--num_workers", "2",
          "--pretrain_choice", "ViT-B/16", "--img_size", "384", "128", "--stride_size", "16", "--img_aug",
          "--loss_names", "sdm+mlm+id", "--MLM", "--lr", str(BACKBONE_LR), "--scfp_lr", str(SCFP_LR),
          "--num_epoch", str(EPOCHS), "--warmup_epochs", str(min(5, max(0, EPOCHS - 1))), "--scfp_diagnostics"]
if TRAINING_PROTOCOL == "warmstart":
    # Match trained IRRA head/backbone shapes and original objective weights.
    for key in ("pretrain_choice", "stride_size", "text_length", "vocab_size", "cmt_depth",
                "loss_names", "id_loss_weight", "mlm_loss_weight", "temperature"):
        if key in original:
            common += ["--" + key, str(original[key])]
    if "img_size" in original:
        common += ["--img_size", *map(str, original["img_size"])]
    if RESUME_CHECKPOINT is None:
        common += ["--init_checkpoint", INIT_CHECKPOINT]
if RESUME_CHECKPOINT:
    common += ["--resume_checkpoint", RESUME_CHECKPOINT]
# The FIRST model experiment is one full V2 batch with the actual IRRA checkpoint.
if RUN_SMOKE_TEST:
    smoke_common = common.copy()
    if RESUME_CHECKPOINT:
        raise ValueError("Set RUN_SMOKE_TEST=False when resuming; smoke does not restore optimizer state")
    smoke_args = [sys.executable, "-m", "experiments.scfp.smoke_test", *smoke_common,
                  "--output_file", str(SESSION / "smoke.json")]
    if TRAINING_PROTOCOL == "warmstart":
        smoke_args += ["--config_file", ORIGINAL_CONFIG]
    run(smoke_args)


In [ ]:
if RUN_BASELINE_EVALUATION:
    run([sys.executable, "test.py", "--config_file", ORIGINAL_CONFIG, "--checkpoint", BASELINE_CHECKPOINT,
         "--root_dir", DATA_ROOT, "--metrics_output", str(SESSION / "baseline_official_test.json")])
# Training only the selected experiment. No automatic 4-mode / 60-epoch sweep.
experiment_root = RUNS / labels[EXPERIMENT]
run([sys.executable, "train.py", *common, "--name", labels[EXPERIMENT], "--output_dir", experiment_root])
if RESUME_CHECKPOINT:
    RUN_DIR = Path(RESUME_CHECKPOINT).resolve().parent
else:
    candidates = sorted((experiment_root / "ICFG-PEDES").glob("*/configs.yaml"), key=lambda p: p.stat().st_mtime)
    assert candidates, "Training produced no run config"
    RUN_DIR = candidates[-1].parent
print("Run directory:", RUN_DIR)
print("Saved artifacts:", [p.name for p in RUN_DIR.iterdir()])


In [ ]:
if RUN_FINAL_TEST:
    # Decide the scientific configuration BEFORE enabling official test.
    chosen = RUN_DIR / ("best.pth" if TRAINING_PROTOCOL == "scientific" else "last.pth")
    run([sys.executable, "test.py", "--config_file", RUN_DIR / "configs.yaml", "--checkpoint", chosen,
         "--root_dir", DATA_ROOT, "--metrics_output", str(SESSION / "final_official_test.json")])
export = RUNS / ("SCFP_V2_" + session_name + ".zip")
with zipfile.ZipFile(export, "x", compression=zipfile.ZIP_STORED) as archive:
    for folder in (RUN_DIR, SESSION):
        for path in folder.rglob("*"):
            if path.is_file():
                archive.write(path, str(path.relative_to(RUNS)))
print("Download/export:", export)
print("Logs/checkpoints:", RUN_DIR)


## Next runs

1. Default: `v2`, warm-start, **1 epoch**. Inspect losses, gradients, frequency variance and slot maps.
2. Set `EPOCHS=5` for a **new** engineering run initialized from the original IRRA checkpoint. This does not establish generalization.
3. Scientific screening: set `TRAINING_PROTOCOL="scientific"`, `INIT_CHECKPOINT=None`, `BACKBONE_LR=1e-5`, `EPOCHS=15`, fixed seed. Run `baseline`, `slots`, `frequency`, `v2` separately with identical data/augmentation/budget. The saved PID manifest must match.
4. Resume: set `RESUME_CHECKPOINT` to the run's `last.pth`, preserve its total EPOCHS and all schedule/split settings, and set `RUN_SMOKE_TEST=False`. Do not resume a 1-epoch schedule as a 5-epoch schedule.
5. Official test is for the decided final configuration. No test score is fabricated or inferred from warm-start holdouts.

`best.pth` is produced only when validation runs. Warm-start with no validation saves `last.pth`, with `best_top1=null`.
The historical user-reported baseline R1=63.573%, mAP=38.149% is a reference, not a result of this notebook.